# LINet Training on Omni RGB-D - Google Colab

**Complete end-to-end training pipeline for Direct Mixing ResNet (LINet) on Google Colab with A100 GPU**

---

## 📋 Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime → Change runtime type → Hardware accelerator: GPU → GPU type: A100
- [ ] **Mount Google Drive:** Your code and dataset will be stored on Drive
- [ ] **Upload dataset to Drive:** `MyDrive/datasets/OmniObject3D_pretrain_256.tar.gz` (preprocessed OmniObject3D dataset)
- [ ] **Expected Runtime:** ~2-3 hours for training

---

## 🎯 What This Notebook Does:

1. ✅ Verify A100 GPU is available
2. ✅ Mount Google Drive
3. ✅ Clone your repository to local RAM (fast I/O)
4. ✅ Copy Omni RGB-D dataset to local RAM (10-20x faster than Drive)
5. ✅ Install dependencies
6. ✅ Train LINet (Direct Mixing ResNet) with all optimizations
7. ✅ Save checkpoints to Drive (persistent storage)
8. ✅ Generate training curves and analysis

---

## 🧠 About LINet:

**LINet** (Direct Mixing Network) is a 2-stream neural network architecture where:
- **RGB stream** processes color images
- **Depth stream** processes depth maps
- **Integrated Stream** combines both streams using learned scalar mixing weights at every layer

Unlike traditional fusion methods, LINet performs integration **inside each convolution neuron** through scalar-based direct mixing:
- Per-stream weights (full kernels for RGB and Depth)
- Integrated weight (1×1 channel-wise for integrated features)
- Scalar mixing coefficients (α, γ) learned per layer to combine stream outputs

This allows the network to learn optimal integration strategies at every layer with minimal computational overhead!

---

**Let's get started!** 🚀

## 1. Environment Setup & GPU Verification

In [1]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    # Check if it's A100
    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\n✅ A100 GPU detected - PERFECT for training!")
    elif 'V100' in gpu_name:
        print("\n✅ V100 GPU detected - Good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\n⚠️  T4 GPU detected - Will be slower, consider upgrading to A100")
    else:
        print(f"\n⚠️  GPU: {gpu_name} - Consider using A100 for best performance")
else:
    print("\n❌ NO GPU DETECTED!")
    print("Please enable GPU: Runtime → Change runtime type → Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

GPU VERIFICATION
PyTorch version: 2.10.0+cu128
CUDA available: True
CUDA version: 12.8
GPU Device: NVIDIA A100-SXM4-80GB
GPU Memory: 79.25 GB

✅ A100 GPU detected - PERFECT for training!



In [2]:
# Detailed GPU info
!nvidia-smi

Tue Mar 24 05:44:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             52W /  400W |       6MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [3]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

Mounted at /content/drive

✅ Google Drive mounted successfully!

Drive contents:
total 3117887
-rw------- 1 root root        176 Sep 21  2019 06-lab2.gdoc
-rw------- 1 root root      21621 Sep 30  2024 113-1363667-3121001@USSR24093000064918@pre-paid.png
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (1).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (2).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final (3).gdoc
-rw------- 1 root root        176 Aug 13  2020 2020 summer final.gdoc
-rw------- 1 root root        176 Jul 11  2025 2025_Gabriel_Clinger_Contractor Agreement_BASE copy.gdoc
-rw------- 1 root root      32204 Apr 18  2022 2900 On First- Welcome Home Next Steps.docx
-rw------- 1 root root       8822 Jun 24  2017 A6.docx
-rw------- 1 root root      22204 Jan 21  2023 activity (1).xlsx
-rw------- 1 root root      22161 Jan 21  2023 activity (2).xlsx
-rw------- 1 root root        176 Jan 21  2023 activity.gsheet
-rw------- 

## 3. Clone Repository to Local Disk (Fast I/O)

**Important:** We clone to `/content/` (local SSD) instead of Drive for 10-20x faster I/O

**Default:** Clone from GitHub (recommended - always gets latest code)

In [4]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"  # UPDATE THIS
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"  # Local copy for fast I/O

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

# Ensure we're in a valid directory
os.chdir('/content')
print(f"Starting in: {os.getcwd()}")

# Check if repo already exists (same session, rerunning cell)
if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"\n📁 Repo already exists: {LOCAL_REPO_PATH}")
    print(f"🔄 Pulling latest changes...")

    os.chdir(LOCAL_REPO_PATH)
    !git pull
    print("✅ Repo updated")

# Clone from GitHub (first run)
else:
    # Remove old incomplete copy if exists
    if Path(LOCAL_REPO_PATH).exists():
        print(f"\n🗑️  Removing incomplete repo copy...")
        !rm -rf {LOCAL_REPO_PATH}

    print(f"\n🔄 Cloning from GitHub...")
    print(f"   Repo: {GITHUB_REPO}")
    print(f"   Destination: {LOCAL_REPO_PATH}")

    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}

    # Verify clone succeeded
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository to {LOCAL_REPO_PATH}")

    print("✅ Repo cloned successfully")
    os.chdir(LOCAL_REPO_PATH)

# Verify repo structure
print(f"\n📂 Repository structure:")
!ls -la {LOCAL_REPO_PATH}

print(f"\n✅ Working directory: {os.getcwd()}")

REPOSITORY SETUP
Starting in: /content

🔄 Cloning from GitHub...
   Repo: https://github.com/clingergab/Multi-Stream-Neural-Networks.git
   Destination: /content/Multi-Stream-Neural-Networks
Cloning into '/content/Multi-Stream-Neural-Networks'...
remote: Enumerating objects: 3144, done.
remote: Counting objects: 100% (167/167), done.
remote: Compressing objects: 100% (70/70), done.
remote: Total 3144 (delta 132), reused 120 (delta 97), pack-reused 2977 (from 2)
Receiving objects: 100% (3144/3144), 113.06 MiB | 37.53 MiB/s, done.
Resolving deltas: 100% (1957/1957), done.
Encountered 49 file(s) that should have been pointers, but weren't:
	tests/augmentation_comparison.png
	tests/augmentation_test.png
	tests/balanced_augmentation_test.png
	tests/balanced_samples_comparison.png
	tests/dataset_orthogonal_loading.png
	tests/decaying_restarts_eta_min_bug.png
	tests/easing_formula_analysis.png
	tests/easing_schedulers_comparison.png
	tests/global_vs_local_comparison.png
	tests/linear_scale_co

## 4. Install Dependencies

In [5]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import ray
import kornia

print("✅ All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   ray: {ray.__version__}")
print(f"   kornia: {kornia.__version__}")


Installing dependencies...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 116.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 MB 36.2 MB/s eta 0:00:00
✅ All dependencies installed!
   h5py: 3.16.0
   matplotlib: 3.10.0
   ray: 2.54.0
   kornia: 0.8.2


## 5. Copy OmniObject3D Dataset to Local Disk

**Performance Note:** Local disk I/O is ~10-20x faster than Drive!

**Dataset:** OmniObject3D pretrain dataset with RGB + Depth (per-sample .pt tensors)

In [6]:
from pathlib import Path
import os

# Paths
DRIVE_DATASET_TAR = "/content/drive/MyDrive/datasets/OmniObject3D_Pretrain_256.tar.gz"
LOCAL_DATASET_PATH = "/dev/shm/sparse_omni_256"  # Extracted location

print("=" * 60)
print("OMNIOBJECT3D PRETRAIN DATASET SETUP (2-STREAM: RGB + DEPTH)")
print("=" * 60)

# Check if already on local disk
if Path(LOCAL_DATASET_PATH).exists() and Path(f"{LOCAL_DATASET_PATH}/class_names.txt").exists():
    print(f"Already on local disk: {LOCAL_DATASET_PATH}")
    # Count samples
    n_pt = len(list(Path(LOCAL_DATASET_PATH).rglob("*_rgb.pt")))
    print(f"   RGB-D pairs: {n_pt}")

# Copy and extract from Drive
elif Path(DRIVE_DATASET_TAR).exists():
    print(f"Found on Drive: {DRIVE_DATASET_TAR}")
    print(f"Copying to local disk...")

    tar_name = Path(DRIVE_DATASET_TAR).name
    local_tar = f"/dev/shm/{tar_name}"

    !rsync -ah --info=progress2 {DRIVE_DATASET_TAR} {local_tar}

    print(f"\nExtracting dataset to local disk...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"

    !rm {local_tar}

    n_pt = len(list(Path(LOCAL_DATASET_PATH).rglob("*_rgb.pt")))
    print(f"Extracted. RGB-D pairs: {n_pt}")

else:
    print(f"Dataset not found on Drive!")
    print(f"   Expected: {DRIVE_DATASET_TAR}")
    print(f"   Run notebooks/omni_preprocess.ipynb to create it.")
    raise FileNotFoundError(f"Dataset not found at {DRIVE_DATASET_TAR}")

print("\n" + "=" * 60)
print(f"Dataset ready at: {LOCAL_DATASET_PATH}")
print("=" * 60)


OMNIOBJECT3D PRETRAIN DATASET SETUP (2-STREAM: RGB + DEPTH)
Found on Drive: /content/drive/MyDrive/datasets/OmniObject3D_Pretrain_256.tar.gz
Copying to local disk...
          9.72G 100%  121.11MB/s    0:01:16 (xfr#1, to-chk=0/1)

Extracting dataset to local disk...
Extracted. RGB-D pairs: 204606

Dataset ready at: /dev/shm/sparse_omni_256


## 6. Setup Python Path & Import LINet

In [7]:
import sys
import os

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Verify project structure
print("Project structure:")
!ls -la {project_root}/src/models/

# Import LiNet and OmniPretrain dataloader
print("\nImporting LiNet and dataloaders...")
from src.models.linear_integration.li_net3 import li_resnet18
from src.data_utils.omnipretrain_dataset import get_omnipretrain_dataloaders
from src.training.augmentation_config import AugmentationConfig


# Import Ray Tune
from ray import train, tune
from ray.tune.schedulers import ASHAScheduler

print("✅ LINet3, dataloaders, and Ray Tune imported successfully!")

Project structure:
total 48
drwxr-xr-x 11 root root 4096 Mar 24 05:44 .
drwxr-xr-x  7 root root 4096 Mar 24 05:44 ..
drwxr-xr-x  2 root root 4096 Mar 24 05:44 abstracts
drwxr-xr-x  2 root root 4096 Mar 24 05:44 common
drwxr-xr-x  2 root root 4096 Mar 24 05:44 core
drwxr-xr-x  2 root root 4096 Mar 24 05:44 direct_mixing_activation
drwxr-xr-x  2 root root 4096 Mar 24 05:44 direct_mixing_bn
drwxr-xr-x  2 root root 4096 Mar 24 05:44 direct_mixing_conv
-rw-r--r--  1 root root 1076 Mar 24 05:44 __init__.py
drwxr-xr-x  4 root root 4096 Mar 24 05:44 linear_integration
drwxr-xr-x  2 root root 4096 Mar 24 05:44 multi_channel
drwxr-xr-x  2 root root 4096 Mar 24 05:44 utils

Importing LiNet and dataloaders...
✅ LINet3, dataloaders, and Ray Tune imported successfully!


## 8b. Hyperparameter Tuning with Ray Tune

Perform a wide search for optimal hyperparameters using Ray Tune.
- **Parallel Trials:** Run multiple configurations simultaneously
- **Short Duration:** Train for limited epochs per trial
- **ASHA Scheduler:** Early-stop unpromising trials
- **Train/Val Split:** Stratified 90/10 split (no k-fold)

In [8]:
import os
import time

# 1. Define Paths explicitly
mps_pipe_dir = "/tmp/nvidia-mps"
mps_log_dir = "/tmp/nvidia-log"

# 2. Create the directories (CRITICAL: Daemon fails if log dir doesn't exist)
os.makedirs(mps_pipe_dir, exist_ok=True)
os.makedirs(mps_log_dir, exist_ok=True)

# 3. Set Environment Variables for the current Python process
os.environ["CUDA_MPS_PIPE_DIRECTORY"] = mps_pipe_dir
os.environ["CUDA_MPS_LOG_DIRECTORY"] = mps_log_dir
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# 4. Configure GPU and Start Daemon using the SAME environment variables
# We use f-strings to pass the python variables into the shell command
print("Setting GPU to Exclusive Process Mode...")
!nvidia-smi -i 0 -c EXCLUSIVE_PROCESS

print("Starting MPS Daemon...")
# We explicitly pass the env vars to the shell command
!export CUDA_MPS_PIPE_DIRECTORY={mps_pipe_dir} && \
 export CUDA_MPS_LOG_DIRECTORY={mps_log_dir} && \
 nvidia-cuda-mps-control -d

# 5. Verify it is running
print("Verifying Daemon Status...")
time.sleep(1) # Give it a second to start
!ps -ef | grep mps

# Check if the pipe file actually exists
if os.path.exists(os.path.join(mps_pipe_dir, "control")):
    print("✅ MPS Control Pipe found. Setup success.")
else:
    print("❌ MPS Control Pipe NOT found. Check /tmp/nvidia-log for errors.")
    # Optional: Print logs if it failed
    !cat {mps_log_dir}/control.log

Setting GPU to Exclusive Process Mode...
Set compute mode to EXCLUSIVE_PROCESS for GPU 00000000:00:05.0.
All done.
Starting MPS Daemon...
Verifying Daemon Status...
root       10216       1  0 05:52 ?        00:00:00 nvidia-cuda-mps-control -d
root       10222    7508  0 05:52 ?        00:00:00 /bin/bash -c ps -ef | grep mps
root       10224   10222  0 05:52 ?        00:00:00 grep mps
✅ MPS Control Pipe found. Setup success.


In [9]:
import random
import numpy as np

import ray
from ray import tune
from ray.tune.schedulers import ASHAScheduler
from ray.tune.search.hyperopt import HyperOptSearch
import torch
from collections import Counter

from src.models.linear_integration.li_net3 import li_resnet18
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.data_utils.omnipretrain_dataset import (
    OmniPretrainDataset,
    _load_class_names,
    _load_norm_stats,
    _discover_samples,
)
from src.training.augmentation_config import AugmentationConfig
from src.utils.seed import set_seed


class TrialTerminated(Exception):
    """Raised when a trial should be terminated early."""
    pass

class RayTuneReporter:
    """Callback for reporting metrics to Ray Tune during training."""

    def __init__(self):
        self.best_accuracy = 0.0
        self.best_loss = float('inf')
        self.best_train_acc = 0.0
        self.best_val_mca = 0.0
        self.best_train_mca = 0.0

    def on_epoch_end(self, epoch, logs):
        """Report current AND best metrics to Ray Tune."""
        if logs['val_accuracy'] > self.best_accuracy:
            self.best_accuracy = logs['val_accuracy']
            if logs['train_accuracy'] > self.best_train_acc:
                self.best_train_acc = logs['train_accuracy']

        if logs['val_loss'] < self.best_loss:
            self.best_loss = logs['val_loss']

        val_mca = logs.get('val_mca', 0.0)
        train_mca = logs.get('train_mca', 0.0)
        if val_mca > self.best_val_mca:
            self.best_val_mca = val_mca
        if train_mca > self.best_train_mca:
            self.best_train_mca = train_mca

        gap = self.best_train_mca - self.best_val_mca
        composite = self.best_val_mca - 10 * (gap**3)

        metrics = {
            "accuracy": logs['val_accuracy'],
            "loss": logs['val_loss'],
            "best_accuracy": self.best_accuracy,
            "best_loss": self.best_loss,
            "train_loss": logs['train_loss'],
            "train_accuracy": logs['train_accuracy'],
            "best_train_acc": self.best_train_acc,
            "val_mca": val_mca,
            "train_mca": train_mca,
            "best_val_mca": self.best_val_mca,
            "best_train_mca": self.best_train_mca,
            "gap": gap,
            "composite": composite,
        }

        tune.report(metrics)


def train_linet_tune(
    config,
    data_root=None,
    norm_stats=None,
    num_classes=None,
    train_samples=None,
    val_samples=None,
    class_names=None,
    seed=42,
):
    """
    Trainable function for Ray Tune.

    Uses pre-split train/val samples (split at preprocessing time by
    object ID to prevent data leakage).

    Args:
        config: Ray Tune configuration dict with hyperparameters
        data_root: Path to dataset root
        norm_stats: Normalization statistics dict
        num_classes: Number of classes
        train_samples: Pre-discovered train (rgb_path, depth_path, label) list
        val_samples: Pre-discovered val (rgb_path, depth_path, label) list
        class_names: List of class name strings
        seed: Random seed for reproducible trials
    """
    set_seed(seed, deterministic=False)
    g = torch.Generator().manual_seed(seed)

    # Per-trial augmentation config
    aug_config = AugmentationConfig(
        rgb_aug_prob=1.0,
        rgb_aug_mag=1.0,
        depth_aug_prob=1.0,
        depth_aug_mag=1.0,
    )

    # Create datasets (train/val already split at preprocessing time)
    train_dataset = OmniPretrainDataset(
        data_root=data_root,
        split='train',
        samples=train_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        normalize=False,  # GPU will normalize after augmentation
        **aug_config.to_dict(),
    )
    val_dataset = OmniPretrainDataset(
        data_root=data_root,
        split='val',
        samples=val_samples,
        class_names=class_names,
        norm_stats=norm_stats,
        normalize=False,
    )

    # Stratified sampling for training
    subset_labels = [s[2] for s in train_samples]
    label_counts = Counter(subset_labels)
    num_samples = len(subset_labels)
    class_weights = {label: num_samples / count for label, count in label_counts.items()}
    sample_weights = torch.tensor(
        [class_weights[label] for label in subset_labels], dtype=torch.float32
    )

    train_sampler = torch.utils.data.WeightedRandomSampler(
        weights=sample_weights,
        num_samples=num_samples,
        replacement=True,
        generator=g,
    )

    def worker_init_fn(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)

    train_loader = torch.utils.data.DataLoader(
        train_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        sampler=train_sampler,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=True,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )
    val_loader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=config['batch_size'],
        shuffle=False,
        num_workers=1,
        prefetch_factor=2,
        persistent_workers=False,
        pin_memory=True,
        worker_init_fn=worker_init_fn,
    )

    # Create Model
    model = li_resnet18(
        num_classes=num_classes,
        stream_input_channels=[3, 1],
        dropout_p=0.3,
        width_multiplier=0.75,
        device="cuda",
        use_amp=True,
    )

    # Create Optimizer
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=[config["lr_rgb"], config["lr_depth"]],
        stream_weight_decays=[config["wd_rgb"], config["wd_depth"]],
        shared_lr=config["lr_shared"],
        integration_weight_decay=config["wd_integrated"],
    )

    # Create Scheduler
    warmup_epochs = 5
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type='cosine',
        eta_min=[1e-6, 1e-6, 1e-6, 1e-6],
        t_max=95,
        train_loader_len=len(train_loader),
        warmup_epochs=warmup_epochs,
        warmup_start_factor=0.2,
    )

    # Compile model
    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=0.1,
        gpu_augmentation=True,
        norm_stats=norm_stats,
        **aug_config.to_dict(),
    )

    # Train
    try:
        model.fit(
            train_loader=train_loader,
            val_loader=val_loader,
            epochs=100,
            early_stopping=True,
            patience=15,
            grad_clip_norm=1.0,
            modality_dropout=True,
            modality_dropout_start=0,
            modality_dropout_ramp=20,
            modality_dropout_rate=0.15,
            callbacks=[RayTuneReporter()],
            verbose=False,
        )
    except TrialTerminated as e:
        print(f"\n{e}")


In [10]:
# =============================================================================
# CONFIGURATION
# =============================================================================
# Ray Tune saves ALL experiment state to DRIVE_STORAGE_PATH via storage_path.
# When Colab dies, re-run the notebook — Tuner.restore() picks up where it
# left off. Completed trials preserved, interrupted trials restart.
# =============================================================================

import hashlib
import json as json_module
import os
import pandas as pd
from pathlib import Path

# --- Ray Tune persistent storage on Google Drive ---
DRIVE_STORAGE_PATH = "/content/drive/MyDrive/ray_tune_experiments"
LOCAL_STORAGE_PATH = "/content/ray_results"
EXPERIMENT_NAME = "omni_pretrain_hpo"


SEED = 42
NUM_SAMPLES = 50  # Total trials to run across all sessions


Path(DRIVE_STORAGE_PATH).mkdir(parents=True, exist_ok=True)
Path(LOCAL_STORAGE_PATH).mkdir(parents=True, exist_ok=True)

experiment_path = os.path.join(DRIVE_STORAGE_PATH, EXPERIMENT_NAME)
local_experiment_path = os.path.join(LOCAL_STORAGE_PATH, EXPERIMENT_NAME)
RESUME_EXISTING = os.path.exists(experiment_path)

if RESUME_EXISTING:
    # Validate experiment dir has actual content
    _exp_files = os.listdir(experiment_path) if os.path.isdir(experiment_path) else []
    if len(_exp_files) == 0:
        print(f"  WARNING: {experiment_path} exists but is empty \u2014 starting fresh")
        RESUME_EXISTING = False

print(f"Drive storage: {DRIVE_STORAGE_PATH}")
print(f"Local storage: {LOCAL_STORAGE_PATH}")
print(f"Experiment: {EXPERIMENT_NAME}")
print(f"Resume existing: {RESUME_EXISTING}")
print(f"Total trials: {NUM_SAMPLES}")
if RESUME_EXISTING:
    print(f"\n  Previous experiment found at {experiment_path}")
    print(f"  Copying Drive -> local, then Tuner.restore() from local.")
    # Copy experiment state from Drive to local before Tuner.restore
    import subprocess as _sp_cfg
    os.makedirs(local_experiment_path, exist_ok=True)
    _result = _sp_cfg.run(
        ["rsync", "-a", experiment_path + "/", local_experiment_path + "/"],
        capture_output=True, text=True,
    )
    if _result.returncode == 0:
        print(f"  Restored to {local_experiment_path}")
    else:
        raise RuntimeError(f"Restore failed: {_result.stderr[:300]}")


Drive storage: /content/drive/MyDrive/ray_tune_experiments
Experiment: omni_pretrain_hpo
Resume existing: False
Total trials: 50


In [ ]:
# Initialize Ray
import shutil
import subprocess
import time as _time

from ray.tune import CLIReporter
from ray.tune import Callback as TuneCallback

os.environ["RAY_AIR_NEW_OUTPUT"] = "0"  # must be set BEFORE ray.init()

class DriveSyncCallback(TuneCallback):
    """Periodically rsyncs local Ray Tune experiment state to Google Drive.

    Ray Tune writes to LOCAL_STORAGE_PATH (fast local disk).  This callback
    rsyncs local -> Drive incrementally (only changed files) so that state
    survives Colab session death without blocking the Ray driver.
    """

    def __init__(self, local_storage_path, drive_storage_path, experiment_name,
                 sync_interval_seconds=300):
        self._local_path = os.path.join(local_storage_path, experiment_name)
        self._drive_path = os.path.join(drive_storage_path, experiment_name)
        self._sync_interval = sync_interval_seconds
        self._last_sync = 0.0

    def _sync(self, reason=""):
        if not os.path.isdir(self._local_path):
            return
        try:
            os.makedirs(self._drive_path, exist_ok=True)
            result = subprocess.run(
                ["rsync", "-a",
                 self._local_path + "/",
                 self._drive_path + "/"],
                capture_output=True, text=True, timeout=120,
            )
            if result.returncode == 0:
                self._last_sync = _time.time()
                print(f"[DriveSyncCallback] synced to Drive ({reason})")
            else:
                print(f"[DriveSyncCallback] WARNING: rsync failed: {result.stderr[:200]}")
        except subprocess.TimeoutExpired:
            print(f"[DriveSyncCallback] WARNING: rsync timed out (120s)")
        except Exception as e:
            print(f"[DriveSyncCallback] WARNING: sync failed: {e}")

    def on_trial_result(self, iteration, trials, trial, result, **info):
        if _time.time() - self._last_sync >= self._sync_interval:
            self._sync(reason=f"periodic, iter={result.get('training_iteration', '?')}")

    def on_trial_complete(self, iteration, trials, trial, **info):
        if _time.time() - self._last_sync >= 60:
            self._sync(reason="trial complete")

    def on_experiment_end(self, trials, **info):
        self._sync(reason="experiment end")


ray.shutdown()
ray.init(
    ignore_reinit_error=True,
    runtime_env={
        "env_vars": {
            "CUDA_MPS_PIPE_DIRECTORY": "/tmp/nvidia-mps",
            "CUDA_MPS_LOG_DIRECTORY": "/tmp/nvidia-log",
            "CUDA_DEVICE_ORDER": "PCI_BUS_ID",
            "CUDA_VISIBLE_DEVICES": "0",
        }
    }
)

# Load dataset metadata (once, shared across all trials)
class_names = _load_class_names(LOCAL_DATASET_PATH)
norm_stats = _load_norm_stats(LOCAL_DATASET_PATH)
train_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'train'), class_names)
val_samples = _discover_samples(os.path.join(LOCAL_DATASET_PATH, 'val'), class_names)
num_classes = len(class_names)

print(f"Dataset: {LOCAL_DATASET_PATH}")
print(f"  Classes: {num_classes}")
print(f"  Train samples: {len(train_samples)}")
print(f"  Val samples: {len(val_samples)}")


# Define trainable (same for both new and restored runs)
trainable = tune.with_resources(
    tune.with_parameters(
        train_linet_tune,
        data_root=LOCAL_DATASET_PATH,
        norm_stats=norm_stats,
        num_classes=num_classes,
        train_samples=train_samples,
        val_samples=val_samples,
        class_names=class_names,
        seed=SEED,
    ),
    resources={"cpu": 1, "gpu": 0.1},
)


# Callback to force-sync experiment state to Drive
drive_sync_cb = DriveSyncCallback(LOCAL_STORAGE_PATH, DRIVE_STORAGE_PATH, EXPERIMENT_NAME)


if RESUME_EXISTING:
    # =========================================================
    # RESUME: Restore previous experiment from Google Drive
    # =========================================================
    # Restores: completed trials, ASHA scheduler state, HyperOpt
    # search algorithm state. Interrupted trials restart from epoch 0.
    print("\n" + "=" * 60)
    print("RESUMING EXPERIMENT FROM GOOGLE DRIVE")
    print("=" * 60)

    tuner = tune.Tuner.restore(
        path=local_experiment_path,
        trainable=trainable,
        resume_unfinished=True,
        resume_errored=True,
    )

else:
    # =========================================================
    # NEW: Create fresh experiment
    # =========================================================
    print("\n" + "=" * 60)
    print("STARTING NEW EXPERIMENT")
    print("=" * 60)

    # Search space
    search_space = {
        # Learning rates
        "lr_rgb": tune.loguniform(1e-5, 5e-4),
        "lr_depth": tune.loguniform(1e-5, 5e-4),
        "lr_shared": tune.loguniform(1e-5, 5e-4),

        # Weight decay
        "wd_rgb": tune.loguniform(1e-6, 5e-4),
        "wd_depth": tune.loguniform(1e-6, 5e-4),
        "wd_integrated": tune.loguniform(1e-5, 1e-3),

        # Scheduler eta_min
        # "s1_eta_min": tune.uniform(5e-8, 5e-6),
        # "s2_eta_min": tune.uniform(5e-8, 5e-6),
        # "eta_min": tune.uniform(1e-8, 1e-6),

        # batch size
        "batch_size": tune.choice([96, 128]),

        # # Regularization
        # "dropout_p": tune.uniform(0.35, 0.55),
        # "label_smoothing": tune.uniform(0.001, 0.15),
        # "grad_clip_norm": tune.uniform(0.5, 1.5),

        # # Augmentation parameters
        # "rgb_aug_prob": tune.uniform(0.9, 1.8),
        # "rgb_aug_mag": tune.uniform(0.9, 1.8),
        # "depth_aug_prob": tune.uniform(0.9, 1.8),
        # "depth_aug_mag": tune.uniform(0.9, 1.8),

        # # Modality dropout
        # "modality_dropout_rate": tune.uniform(0.12, 0.17),
    }



    reporter = CLIReporter(
        parameter_columns=[
            "lr_rgb", "lr_depth", "lr_shared",
            "wd_rgb", "wd_depth", "wd_integrated",
            "batch_size",
            # "dropout_p", "label_smoothing", "grad_clip_norm",
            # "rgb_aug_prob", "rgb_aug_mag",
            # "depth_aug_prob", "depth_aug_mag",
            # "modality_dropout_rate", "modality_dropout_start", "modality_dropout_ramp",
        ],
        metric_columns={
            "training_iteration": "iter",
            "best_val_mca": "best_val_mca",
            "best_train_mca": "best_train_mca",
            "best_accuracy": "best_accuracy",
            "composite": "composite",
        },
        max_report_frequency=30,
        print_intermediate_tables=True,
    )

    hyperopt_search = HyperOptSearch(
        metric="composite",
        mode="max",
        n_initial_points=15  # <--- Your precise warmup tweak
    )

    asha_scheduler = ASHAScheduler(
        time_attr="training_iteration",
        metric="composite",
        mode="max",
        max_t=100,
        grace_period=15,
        reduction_factor=2,
    )

    tuner = tune.Tuner(
        trainable,
        param_space=search_space,
        tune_config=tune.TuneConfig(
            scheduler=asha_scheduler,
            search_alg=hyperopt_search,
            num_samples=NUM_SAMPLES,
            max_concurrent_trials=10,
        ),
        run_config=ray.tune.RunConfig(
            storage_path=LOCAL_STORAGE_PATH,
            name=EXPERIMENT_NAME,
            progress_reporter=reporter,
            verbose=1,
            callbacks=[drive_sync_cb],
        ),
    )


# Run (or resume) tuning
print("\n" + "=" * 60)
print("STARTING HYPERPARAMETER TUNING")
print("=" * 60)

results = tuner.fit()

best_result = results.get_best_result("best_val_mca", "max")

print("\n" + "=" * 60)
print("TUNING COMPLETE")
print("=" * 60)
print(f"Best Trial Config: {best_result.config}")
print(f"Best Trial Val MCA: {best_result.metrics['best_val_mca']:.4f}")
    print(f"Best Trial Accuracy: {best_result.metrics['best_accuracy']:.4f}")
print(f"Best Trial Loss: {best_result.metrics['best_loss']:.4f}")
print(f"\nExperiment saved to: {local_experiment_path}")
print(f"Drive backup: {experiment_path}")
print(f"To resume after Colab dies: just re-run this notebook.")


2026-03-24 05:52:43,135	INFO worker.py:2013 -- Started a local Ray instance.
/usr/local/lib/python3.12/dist-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Streaming output truncated to the last 5000 lines.
| train_linet_tune_6d645755 | RUNNING  | 172.28.0.12:11546 | 8.09958e-05 | 0.000285499 | 2.38958e-05 | 9.78294e-06 | 2.82721e-06 |     2.46031e-05 |          128 |      1 |       0.100191  |        0.0367536 |   0.102743  |
| train_linet_tune_fb563a56 | RUNNING  | 172.28.0.12:11660 | 0.000196748 | 1.07461e-05 | 8.58074e-05 | 7.50792e-06 | 0.00049589  |     2.25735e-05 |          128 |      1 |       0.167245  |        0.0551359 |   0.181335  |
| train_linet_tune_dc125a7b | RUNNING  | 172.28.0.12:11769 | 2.36375e-05 | 1.57552e-05 | 3.48293e-05 | 4.47761e-06 | 1.97705e-06 |     1.52913e-05 |           96 |      1 |       0.0720884 |        0.0321051 |   0.0727276 |
| train_linet_tune_ae8b175c | RUNNING  | 172.28.0.12:11895 | 2.17281e-05 | 1.45666e-05 | 5.45134e-05 | 0.000242177 | 1.57499e-05 |     4.99391e-05 |           96 |      1 |       0.0932017 |        0.038562  |   0.094833  |
| train_linet_tune_140d4ef0 | RUNNING  | 172.28.0.12:

(train_linet_tune pid=11124) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=11124)   scheduler.step()


== Status ==
Current time: 2026-03-24 08:12:38 (running for 02:19:48.40)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: None
Logical resource usage: 10.0/12 CPUs, 0.9999999999999999/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-03-24_05-52-36_722561_7508/artifacts/2026-03-24_05-52-48/omni_pretrain_hpo/driver_artifacts
Number of trials: 10/50 (10 RUNNING)
+---------------------------+----------+-------------------+-------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------+-----------------+------------------+-------------+
| Trial name                | status   | loc               |      lr_rgb |    lr_depth |   lr_shared |      wd_rgb |    wd_depth |   wd_integrated |   batch_size |   iter |   best_accuracy |   best_train_acc |   composite |
|---------------------------+----------+-------------------+-------------+-------------+-------------+------------

(train_linet_tune pid=11329) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=11329)   scheduler.step()


...

== Status ==
Current time: 2026-03-24 08:39:12 (running for 02:46:21.81)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: None
Logical resource usage: 10.0/12 CPUs, 0.9999999999999999/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-03-24_05-52-36_722561_7508/artifacts/2026-03-24_05-52-48/omni_pretrain_hpo/driver_artifacts
Number of trials: 10/50 (10 RUNNING)
+---------------------------+----------+-------------------+-------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------+-----------------+------------------+-------------+
| Trial name                | status   | loc               |      lr_rgb |    lr_depth |   lr_shared |      wd_rgb |    wd_depth |   wd_integrated |   batch_size |   iter |   best_accuracy |   best_train_acc |   composite |
|---------------------------+----------+-------------------+-------------+-------------+-------------+------------

(train_linet_tune pid=11895) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=11895)   scheduler.step()


Streaming output truncated to the last 5000 lines.
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------+-----------------+------------------+-------------+


== Status ==
Current time: 2026-03-24 13:07:45 (running for 07:14:54.64)
Using AsyncHyperBand: num_stopped=1
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: 0.9775080226698983
Logical resource usage: 10.0/12 CPUs, 0.9999999999999999/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-03-24_05-52-36_722561_7508/artifacts/2026-03-24_05-52-48/omni_pretrain_hpo/driver_artifacts
Number of trials: 11/50 (10 RUNNING, 1 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------+-----------------+------------------+-------------+
| Trial name               

(train_linet_tune pid=116350) /content/Multi-Stream-Neural-Networks/src/training/schedulers.py:193: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
(train_linet_tune pid=116350)   scheduler.step()


== Status ==
Current time: 2026-03-24 14:45:27 (running for 08:52:37.06)
Using AsyncHyperBand: num_stopped=3
Bracket: Iter 60.000: None | Iter 30.000: None | Iter 15.000: 0.9768068619878933
Logical resource usage: 10.0/12 CPUs, 0.9999999999999999/1 GPUs (0.0/1.0 accelerator_type:A100)
Result logdir: /tmp/ray/session_2026-03-24_05-52-36_722561_7508/artifacts/2026-03-24_05-52-48/omni_pretrain_hpo/driver_artifacts
Number of trials: 13/50 (10 RUNNING, 3 TERMINATED)
+---------------------------+------------+--------------------+-------------+-------------+-------------+-------------+-------------+-----------------+--------------+--------+-----------------+------------------+-------------+
| Trial name                | status     | loc                |      lr_rgb |    lr_depth |   lr_shared |      wd_rgb |    wd_depth |   wd_integrated |   batch_size |   iter |   best_accuracy |   best_train_acc |   composite |
|---------------------------+------------+--------------------+-------------+---

In [ ]:
# =============================================================================
# SAVE RESULTS CSV (for offline analysis)
# =============================================================================
# Ray Tune already saved everything to DRIVE_STORAGE_PATH.
# This cell just exports a clean CSV for easy analysis.
# =============================================================================

import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

results_df = results.get_dataframe()

csv_dir = f"{DRIVE_STORAGE_PATH}/analysis"
Path(csv_dir).mkdir(parents=True, exist_ok=True)

csv_path = f"{csv_dir}/omni_hpo_results_{timestamp}.csv"
results_df.to_csv(csv_path, index=False)
print(f"Results CSV saved: {csv_path}")
print(f"  Trials: {len(results_df)}")

# Also save a latest copy
latest_path = f"{csv_dir}/omni_hpo_results_latest.csv"
results_df.to_csv(latest_path, index=False)
print(f"Latest copy: {latest_path}")


In [ ]:
# Analyze Top 10 Trials from Ray Tune (ranked by best_accuracy)
import pandas as pd

print("=" * 80)
print("TOP 10 TRIALS BY BEST VAL MCA (CONTINUOUS SEARCH)")
print("=" * 80)

df = results.get_dataframe()

df_sorted = df.sort_values('best_val_mca', ascending=False)

display_cols = [
    'best_val_mca', 'best_train_mca', 'best_accuracy', 'best_train_acc', 'composite',
    'config/lr_rgb', 'config/lr_depth', 'config/lr_shared',
    'config/wd_rgb', 'config/wd_depth', 'config/wd_integrated',
    'config/batch_size',
    # 'config/s1_eta_min', 'config/s2_eta_min', 'config/eta_min', #'config/t_max',
    # 'config/dropout_p', 'config/label_smoothing', 'config/grad_clip_norm',
    # 'config/rgb_aug_prob', 'config/rgb_aug_mag',
    # 'config/depth_aug_prob', 'config/depth_aug_mag',
    # 'config/modality_dropout_rate', 'config/modality_dropout_start', 'config/modality_dropout_ramp',
]

top_10 = df_sorted[display_cols].head(10)

top_10_formatted = top_10.copy()
top_10_formatted['best_val_mca'] = top_10_formatted['best_val_mca'].apply(lambda x: f"{x*100:.2f}%")
    top_10_formatted['best_train_mca'] = top_10_formatted['best_train_mca'].apply(lambda x: f"{x*100:.2f}%")
    top_10_formatted['best_accuracy'] = top_10_formatted['best_accuracy'].apply(lambda x: f"{x*100:.2f}%")

sci_cols = [
    'config/lr_rgb', 'config/lr_depth', 'config/lr_shared',
    'config/wd_rgb', 'config/wd_depth', 'config/wd_integrated',
    'config/batch_size',
    # 'config/s1_eta_min', 'config/s2_eta_min', 'config/eta_min',
]
for col in sci_cols:
    if col in top_10_formatted.columns:
        top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.2e}")

# float_cols = [
#     'config/dropout_p', 'config/label_smoothing', 'config/grad_clip_norm',
#     'config/rgb_aug_prob', 'config/rgb_aug_mag',
#     'config/depth_aug_prob', 'config/depth_aug_mag',
#     'config/modality_dropout_rate',
# ]
# for col in float_cols:
#     if col in top_10_formatted.columns:
#         top_10_formatted[col] = top_10_formatted[col].apply(lambda x: f"{x:.3f}")

print(top_10_formatted.to_string(index=False))
print("\n" + "=" * 80)


In [ ]:
# =============================================================================
# ANALYZE TOP 10 TRIALS BY COMPOSITE
# =============================================================================

import pandas as pd

df = results.get_dataframe()
df = df.sort_values("composite", ascending=False)
top_10 = df.head(10).copy()

config_cols = [c for c in df.columns if c.startswith("config/")]

print("=" * 80)
print("TOP 10 TRIALS BY COMPOSITE")
print("=" * 80)

for rank, (_, row) in enumerate(top_10.iterrows(), 1):
    gap = row.get("best_train_mca", 0) - row.get("best_val_mca", 0)
    print(f"\n--- #{rank} | Composite: {row['composite']*100:.2f}% | "
          f"Val MCA: {row['best_val_mca']*100:.2f}% | Acc: {row['best_accuracy']*100:.2f}% | "
          f"Gap: {gap*100:.1f}pp ---")

print("\n" + "=" * 80)
print("HYPERPARAMETER RANGES ACROSS TOP 10")
print("=" * 80)
print(f"{'Parameter':<35} {'Min':>12} {'Max':>12} {'Median':>12}")
print("-" * 75)

for col in config_cols:
    short_name = col.replace("config/", "")
    col_min = top_10[col].min()
    col_max = top_10[col].max()
    col_med = top_10[col].median()
    if abs(col_med) < 0.001:
        print(f"{short_name:<35} {col_min:>12.2e} {col_max:>12.2e} {col_med:>12.2e}")
    else:
        print(f"{short_name:<35} {col_min:>12.4f} {col_max:>12.4f} {col_med:>12.4f}")
